# CapitalEdge Analytics - Data Transformation

## Objective
To transform raw JSON stock data into a structured format and implement incremental loading.

## Author: WAETSI Anyanwu

### Imports

In [1]:
import json
import pandas as pd
import os

### Load Latest Data

In [2]:
# Get list of raw files
files = os.listdir("data/raw")

# Pick latest file
latest_file = sorted(files)[-1]

file_path = f"data/raw/{latest_file}"

print(f"Using file: {file_path}")

Using file: data/raw/AAPL_20260415_091353.json


### Load JSON

In [3]:
with open(file_path, "r") as f:
    data = json.load(f)

data.keys()

dict_keys(['Meta Data', 'Time Series (Daily)'])

### Convert to DataFrame

In [18]:
time_series = data["Time Series (Daily)"]

df = pd.DataFrame.from_dict(time_series, orient="index")
df.head()

,1. open,2. high,3. low,4. close,5. volume
2026-04-14,259.2450,261.9300,257.1900,258.8300,48370710
2026-04-13,259.7300,260.1800,256.6600,259.2000,36234698
2026-04-10,259.9800,262.1900,259.0231,260.4800,31291473
2026-04-09,259.0000,261.1200,256.0700,260.4900,28121574
2026-04-08,258.4500,259.7499,256.5300,258.9000,41032772


### Clean Data

In [19]:
# Start fresh and convert index into a column
df = df.reset_index()

# Rename the index column to date
df = df.rename(columns={"index": "date"})

# Convert date column
df["date"] = pd.to_datetime(df["date"])

# Convert all non-date columns to numeric
for col in df.columns:
    if col != "date":
        df[col] = pd.to_numeric(df[col], errors="coerce")

df.head()

,date,1. open,2. high,3. low,4. close,5. volume
0,2026-04-14,259.245,261.9300,257.1900,258.83,48370710
1,2026-04-13,259.730,260.1800,256.6600,259.20,36234698
2,2026-04-10,259.980,262.1900,259.0231,260.48,31291473
3,2026-04-09,259.000,261.1200,256.0700,260.49,28121574
4,2026-04-08,258.450,259.7499,256.5300,258.90,41032772


In [20]:
# check for duplicates
print(df.columns.tolist())
print(df.duplicated().sum())

['date', '1. open', '2. high', '3. low', '4. close', '5. volume']
0


#### Data Transformation and Structure Summary

The raw JSON data was successfully transformed into a structured DataFrame format, making it suitable for analysis and downstream processing.

Key steps included:
- Converting nested JSON into tabular format
- Standardizing column names
- Converting data types appropriately
- Sorting the dataset by date

This step ensures the dataset is clean, structured, and ready for further processing.

### Sorting

In [23]:
df.sort_values(by="date", inplace=True)
df.head()
df.tail()

,date,1. open,2. high,3. low,4. close,5. volume
4,2026-04-08,258.450,259.7499,256.5300,258.90,41032772
3,2026-04-09,259.000,261.1200,256.0700,260.49,28121574
2,2026-04-10,259.980,262.1900,259.0231,260.48,31291473
1,2026-04-13,259.730,260.1800,256.6600,259.20,36234698
0,2026-04-14,259.245,261.9300,257.1900,258.83,48370710


### Incremental Loading

In [21]:
import os

processed_path = "data/processed/stock_data.csv"

# Create folder if needed
os.makedirs("data/processed", exist_ok=True)

if os.path.exists(processed_path):
    existing_df = pd.read_csv(processed_path)
    existing_df["date"] = pd.to_datetime(existing_df["date"])

    # Find new records only
    new_data = df[~df["date"].isin(existing_df["date"])]

    print(f"New records found: {len(new_data)}")

    # Append new data
    updated_df = pd.concat([existing_df, new_data])

else:
    print("No existing file found, creating new dataset")
    updated_df = df

# Save updated dataset
updated_df.to_csv(processed_path, index=False)

print("Incremental update complete ✅")

No existing file found, creating new dataset
Incremental update complete ✅


##### Incremental Loading Summary

Incremental loading was implemented to ensure that only new records are added to the dataset.

On the first run, the system creates a new dataset. On subsequent runs, only new data based on the date field is appended.

This prevents duplication, reduces storage usage, and improves efficiency, making the pipeline scalable for real-world financial data processing.